# Example notebook for an Fst statistic analysis

The present notebook serves as a guide of how to use the `IDEAL-GENOM-QC` library to compute Fst (fixation index) statistics between the study population and each 1000 Genomes super-population.

The underlying module (`ideal_genom.population.fst_stats`) provides the `FstSummary` class. It reuses `ReferenceGenomicMerger` (the same class used by `AncestryQC`) to harmonize and merge the study data with the 1000 Genomes reference panel, tags every sample with a `SuperPop` (or `StPop` for study samples without a reference match), and then runs **PLINK1.9**'s `--fst --within` for each reference super-population against the study population.

Let us import the required libraries.

In [1]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.population.fst_stats import FstSummary

In the next cell the path variables associated with the project are set.

As with ancestry QC and dimensionality reduction, this step is typically run on the cleaned output of the sample QC pipeline. Since each user can have a slightly different choice for the LD regions, the user can provide its own file; otherwise it is fetched automatically for builds **GRCh37** and **GRCh38**.

The user can also provide the path to the reference genome files (`reference_files` dict with `bed`/`bim`/`fam`/`psam` keys) or let the library fetch the 1000 Genomes data automatically (the default, used here).

In [2]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

example_data = DATA_PATH / 'example_data'
ouputData    = example_data / 'outputData'

# Use the cleaned output of the variant QC notebook as input
input_path = ouputData / 'variant_qc_results' / 'clean_files'
input_name = '1KG_GRCh38_variant_qc'
output_path = ouputData
high_ld_file = Path('path/to/ld-regions/file') # if not available, set to Path()

Initialize the class `FstSummary`. Since no `reference_files` are provided, the 1000 Genomes reference panel will be fetched automatically for the chosen build.

If `recompute_merge` is `False`, the merging step below will be skipped and the merged `PLINK` files are expected to already exist under `fst_summary.merging_dir` (e.g. from a previous run).

In [3]:
fst_summary = FstSummary(
    input_path     =input_path,
    input_name     =input_name,
    output_path    =output_path,
    high_ld_file   =high_ld_file,
    build          ='38', # '38' it is the default value
    recompute_merge=True, # if True, it will recompute the merge of the input files
)

INFO:ideal_genom.population.fst_stats:High LD file not found at path/to/ld-regions/file
INFO:ideal_genom.population.fst_stats:High LD file will be fetched from the package
INFO:ideal_genom.population.fst_stats:High LD file fetched from the package and saved at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/ld_regions_files/high-LD-regions_GRCH38.txt
INFO:ideal_genom.population.fst_stats:No reference files provided. Fetching 1000 Genomes reference data for build 38
INFO:ideal_genom.core.get_references:Destination folder: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38
INFO:ideal_genom.core.get_references:1000 Genomes binaries already exist. Skipping download.
INFO:ideal_genom.core.get_references:1000 Genomes binaries already exist. Skipping conversion into bfiles...


`merge_reference_study()` merges the study data with the 1000 Genomes reference panel using the same harmonization steps as `AncestryQC`: renaming SNP IDs, filtering strand-ambiguous SNPs, LD pruning, fixing chromosome/position/allele-flip mismatches, and finally merging. `ind_pair` are the `--indep-pairwise` parameters used for LD pruning.

This step shells out to PLINK many times, which prints a lot of console text; we capture it into `fst_merge_log` to keep the notebook readable — run `fst_merge_log.show()` in a new cell if you need to inspect it.

In [4]:
%%capture fst_merge_log
fst_summary.merge_reference_study(ind_pair=[50, 5, 0.2])

INFO:ideal_genom.qc.ancestry_qc:High LD file found at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/ld_regions_files/high-LD-regions_GRCH38.txt
INFO:ideal_genom.qc.ancestry_qc:Successfully validated 4 reference files
INFO:ideal_genom.qc.ancestry_qc:User-provided reference files validation successful
INFO:ideal_genom.qc.ancestry_qc:STEP: Renaming SNP IDs in the study data using PLINK2
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc --set-all-var-ids @:#:$r:$a --threads 30 --make-bed --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-renamed
INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.qc.ancestry_qc:STEP: Filtering A->T and C->G SNPs from study and reference data.


PLINK v2.0.0-a.6.26LM AVX2 Intel (26 Oct 2025)      cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-renamed.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc
  --make-bed
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-renamed
  --set-all-var-ids @:#:$r:$a
  --threads 30

Start time: Fri Jun 26 13:02:19 2026
63862 MiB RAM detected, ~49680 available; reserving 31931 MiB for main
workspace.
Using up to 30 threads (change this with --threads).
200 samples (94 females, 106 males; 170 founders) loaded from
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc.fam.

INFO:ideal_genom.qc.ancestry_qc:STEP: Filtering problematic SNPs from the study data: 12271 SNPs filtered
INFO:ideal_genom.qc.ancestry_qc:STEP: Filtering problematic SNPs from the reference data: 10178342 SNPs filtered
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-renamed --chr 1-22 --exclude /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc.ac_get_snps --threads 30 --make-bed --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-no_ac_gt_snps --memory 32434
INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38/1kG_phase3_GRCh38 --max-alleles 2 --chr 1-22 --exclude /home/luis/CGE/ideal-genom-qc/ideal_genom/d

PLINK v2.0.0-a.6.26LM AVX2 Intel (26 Oct 2025)      cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-no_ac_gt_snps.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-renamed
  --chr 1-22
  --exclude /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc.ac_get_snps
  --make-bed
  --memory 32434
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-no_ac_gt_snps
  --threads 30

Start time: Fri Jun 26 13:03:00 2026
63862 MiB RAM detected, ~49744 available; reserving 32434 MiB for main
workspace.
Using up to 30 threads (change this with --threads).
200 samples (94 females, 106 males; 170 foun

INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.qc.ancestry_qc:STEP: LD-based pruning of study and reference data
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-no_ac_gt_snps --exclude range /home/luis/CGE/ideal-genom-qc/ideal_genom/data/ld_regions_files/high-LD-regions_GRCH38.txt --keep-allele-order --indep-pairwise 50 5 0.2 --threads 30 --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc
INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-no_ac_gt_snps --extract /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc.prune.in --keep-a

...

686970717273747576777879808182838485868788899091929394949596979899done.
End time: Fri Jun 26 13:03:49 2026


INFO:ideal_genom.qc.ancestry_qc:STEP: Fixing chromosome mismatch between study data and reference panel: 0 SNPs to update
INFO:ideal_genom.qc.ancestry_qc:No chromosome mismatches found. Skipping chromosome update step.
INFO:ideal_genom.qc.ancestry_qc:STEP: Fixing position mismatch between study data and reference panel
INFO:ideal_genom.qc.ancestry_qc:STEP: Fixing position mismatch between study data and reference panel: 0 SNPs to update
INFO:ideal_genom.qc.ancestry_qc:No position mismatches found. Skipping position update step.
INFO:ideal_genom.qc.ancestry_qc:STEP: Allele flipping between study data and reference panel
INFO:ideal_genom.qc.ancestry_qc:STEP: Allele flipping between study data and reference panel: 0 SNPs to flip
INFO:ideal_genom.qc.ancestry_qc:No SNPs require allele flipping. Skipping flipping step.
INFO:ideal_genom.qc.ancestry_qc:STEP: Removing mismatched SNPs from reference data
INFO:ideal_genom.qc.ancestry_qc:STEP: Removing mismatched SNPs from reference data: 0 SNPs t

PLINK v1.90b7.4 64-bit (18 Aug 2024)           www.cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1KG_GRCh38_variant_qc-pruned
  --bmerge /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1kG_phase3_GRCh38-pruned.bed /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1kG_phase3_GRCh38-pruned.bim /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/1kG_phase3_GRCh38-pruned.fam
  --keep-allele-order
  --make-bed
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged
  --threads 30

63862 MB RAM dete

INFO:ideal_genom.core.executor:Command completed successfully


Merged fileset written to                     
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged-merge.bed
+
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged-merge.bim
+
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged-merge.fam
.
44280 variants loaded from .bim file.
3212 people (1603 males, 1608 females, 1 ambiguous) loaded from .fam.
Ambiguous sex ID written to
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged.nosex
.
200 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2591 founders and 621 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132333435363738394041424344454647484950515253545556575859606162636465

In [5]:
print(f"Merging completed. Merged PLINK files written to: {fst_summary.merging_dir}")

Merging completed. Merged PLINK files written to: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging


`add_population_tags()` reads the reference panel's `PSAM` file to tag every reference sample with its `SuperPop`, and labels every study sample not present in the reference panel as `'StPop'`. The result is written to `fst_summary.population_tags`.

In [6]:
fst_summary.add_population_tags()

INFO:ideal_genom.population.fst_stats:Population tags loaded from /home/luis/CGE/ideal-genom-qc/ideal_genom/data/1000genomes_build_38/1kG_phase3_GRCh38.psam
INFO:ideal_genom.population.fst_stats:Population tags columns: ['ID1', 'ID2', 'SuperPop']
INFO:ideal_genom.population.fst_stats:Merged BIM file loaded from /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged.fam
INFO:ideal_genom.population.fst_stats:Merged BIM file columns: ['ID1', 'ID2', 2, 3, 4, 5]
INFO:ideal_genom.population.fst_stats:Added population tags to the merged dataset


In [7]:
population_tags = pd.read_csv(fst_summary.population_tags, sep='\t')
population_tags['SuperPop'].value_counts()

SuperPop
AFR      893
EUR      633
SAS      601
EAS      585
AMR      490
StPop     10
Name: count, dtype: int64

`compute_fst()` reads the population tags and, for each reference `SuperPop` (excluding `'StPop'`), builds `--keep`/`--within` filter files and runs PLINK1.9's `--fst --within` to compare that super-population against the study population.

This also shells out to PLINK repeatedly; we capture its console output into `fst_compute_log` — run `fst_compute_log.show()` in a new cell if you need to inspect it.

In [12]:
log_path = "logs/fst_compute.log"

with open(log_path, "w") as f:
    old_stdout_fd = os.dup(1)
    old_stderr_fd = os.dup(2)
    
    try:
        os.dup2(f.fileno(), 1)
        os.dup2(f.fileno(), 2)

        fst_summary.compute_fst()

    finally:
        os.dup2(old_stdout_fd, 1)
        os.dup2(old_stderr_fd, 2)
        os.close(old_stdout_fd)
        os.close(old_stderr_fd)

print(f"Pipeline complete. Log saved to: {log_path}")

INFO:ideal_genom.population.fst_stats:Created keep and within files for population EUR
INFO:ideal_genom.population.fst_stats:Created keep and within files for population EAS
INFO:ideal_genom.population.fst_stats:Created keep and within files for population AMR
INFO:ideal_genom.population.fst_stats:Created keep and within files for population SAS
INFO:ideal_genom.population.fst_stats:Created keep and within files for population AFR
INFO:ideal_genom.core.executor:Executing: plink --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/merging/cleaned-with-ref-merged --keep /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/keep-EUR_StPop.txt --threads 30 --memory 32369 --make-bed --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/keep-EUR-StPop
INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.core.executor:Executing: plink --bfile /home/luis/CGE/ideal-gen

Pipeline complete. Log saved to: logs/fst_compute.log


In [13]:
print(f"Fst computation completed. Results written to: {fst_summary.results_dir}")

Fst computation completed. Results written to: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results


`report_fst()` parses the `Mean Fst`/`Weighted Fst` lines out of each `fst-{SuperPop}-StPop.log` file, assembles them into a summary `DataFrame`, writes it to `fst_summary.csv` in the results directory, and removes the leftover intermediate `PLINK` binaries (`.bed`/`.bim`/`.fam`) from the results directory.

In [14]:
fst_report = fst_summary.report_fst()
fst_report

INFO:ideal_genom.population.fst_stats:Found 10 log files in /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results
INFO:ideal_genom.population.fst_stats:Found Fst result file for population AMR at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/fst-AMR-StPop.log
INFO:ideal_genom.population.fst_stats:Found Fst result file for population EUR at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/fst-EUR-StPop.log
INFO:ideal_genom.population.fst_stats:Found Fst result file for population EAS at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/fst-EAS-StPop.log
INFO:ideal_genom.population.fst_stats:Found Fst result file for population AFR at /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/fst-AFR-StPop.log
INFO:ideal_genom.population.fst_stats:Found Fst result file for population SAS at /home/luis/CGE/ideal-genom-qc/ideal_genom/d

,SuperPop,Fst,WeightedFst
0,AMR,0.0923244,0.118945
1,EUR,0.124868,0.138953
2,EAS,0.186772,0.174171
3,AFR,-0.00433644,-0.000266101
4,SAS,0.0918748,0.125825


The same summary is available on disk, in case you want to load it again without re-running the pipeline.

**Note:** unlike sample/variant/ancestry QC, there is no dedicated clean-up class for this module — the `keep-*`/`within-*` filter files and PLINK `.log` files are left in `fst_summary.results_dir` for inspection.

In [15]:
fst_csv_path = fst_summary.results_dir / 'fst_summary.csv'
print(f"Fst summary report: {fst_csv_path} (exists: {fst_csv_path.exists()})")

Fst summary report: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/fst_results/fst_summary.csv (exists: True)
